# Heritage Twin - Train YOLOv8-seg phát hiện vết nứt

Notebook này train 1 model phát hiện/khoanh vùng vết nứt (crack segmentation) bằng YOLOv8-seg, độc lập với
pipeline dựng 3D (`heritage_twin_pipeline_colab.ipynb`).

```text
Dataset DeepCrack (mask PNG) -> convert sang định dạng YOLO segmentation -> train YOLOv8-seg
-> đánh giá trên test set -> chạy demo trên ảnh thật -> xuất weight best.pt
```

**Vì sao segmentation chứ không phải object detection (bounding box):** vết nứt là hình mảnh, kéo dài —
khoanh bằng ô vuông (bbox) sẽ chứa rất nhiều vùng không phải vết nứt, nhìn không thuyết phục. Segmentation
khoanh đúng hình dạng thật của vết nứt.

**Vì sao dataset DeepCrack:** đây là dataset crack segmentation có mask ở mức pixel, công khai, và có ảnh
chụp tường/công trình (không chỉ mặt đường như đa số dataset crack khác) nên hợp domain hơn với di sản.

**Giới hạn cần biết:** DeepCrack vẫn chủ yếu là bê tông/tường hiện đại, không phải đá/gỗ/gạch cổ của di
tích Việt Nam — sẽ có độ lệch domain nhất định. Model này đủ tốt để demo khái niệm (proof of concept),
chưa nên tuyên bố là "chính xác cho di sản Việt Nam" nếu chưa test/fine-tune thêm trên ảnh di tích thật.

Cũng lưu trữ toàn bộ dữ liệu trung gian (dataset đã convert, checkpoint, weight) vào Google Drive như
notebook pipeline 3D, để resume được nếu Colab ngắt session.

In [ ]:
# =============================
# 1. CONFIG - chỉnh cell này
# =============================

RUN_NAME = "crack_yolov8n_seg"  # đổi tên nếu muốn thử nghiệm nhiều cấu hình song song

# Model YOLOv8-seg pretrained làm điểm khởi đầu (transfer learning từ COCO)
# "yolov8n-seg.pt" = nano, nhanh nhất, hợp Colab T4. "yolov8s-seg.pt" chính xác hơn nhưng train lâu hơn.
BASE_MODEL = "yolov8n-seg.pt"

IMG_SIZE = 640
EPOCHS = 150            # dataset nhỏ (~300 ảnh train) nên cần nhiều epoch hơn dataset lớn; 150 là mức hợp lý cho prototype
BATCH = -1               # -1 = để ultralytics tự chọn batch size theo VRAM còn trống
VAL_SPLIT_RATIO = 0.15   # trích 15% từ tập train gốc làm val theo dõi trong lúc train; test set giữ nguyên, không đụng tới

# Cờ ép chạy lại (mặc định False = tự động bỏ qua / resume bước đã có, giống notebook pipeline 3D)
FORCE_REDOWNLOAD_DATASET = False
FORCE_RECONVERT_LABELS = False
FORCE_RETRAIN = False

# Paths - toàn bộ dữ liệu trung gian nằm trong Google Drive để không mất khi Colab ngắt session
DRIVE_ROOT = "/content/drive/MyDrive/HeritageTwin"
RAW_DATASET_DIR = f"{DRIVE_ROOT}/crack_detection/raw_deepcrack"
YOLO_DATASET_DIR = f"{DRIVE_ROOT}/crack_detection/yolo_dataset"
TRAIN_PROJECT_DIR = f"{DRIVE_ROOT}/crack_detection/runs"
DEMO_OUTPUT_DIR = f"{DRIVE_ROOT}/crack_detection/demo_outputs"

DATA_YAML_PATH = f"{YOLO_DATASET_DIR}/data.yaml"

print("Run name:", RUN_NAME)
print("Raw dataset dir (Drive):", RAW_DATASET_DIR)
print("YOLO dataset dir (Drive):", YOLO_DATASET_DIR)
print("Train project dir (Drive):", TRAIN_PROJECT_DIR)

In [ ]:
# =============================
# 2. Mount Google Drive
# =============================
from google.colab import drive
drive.mount('/content/drive')

import os
for d in [RAW_DATASET_DIR, YOLO_DATASET_DIR, TRAIN_PROJECT_DIR, DEMO_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)
print("Đã tạo các thư mục làm việc trên Drive.")

In [ ]:
# =============================
# 3. Kiểm tra GPU
# =============================
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# =============================
# 4. Cài đặt thư viện
# =============================
!pip install -q ultralytics opencv-python pillow pyyaml

import ultralytics
ultralytics.checks()

In [ ]:
# =============================
# 5. Tải dataset DeepCrack -> Drive (bỏ qua nếu đã tải)
# =============================
import os, subprocess
from pathlib import Path

zip_path = Path(RAW_DATASET_DIR) / "DeepCrack.zip"
extracted_marker = Path(RAW_DATASET_DIR) / "train_img"

if extracted_marker.exists() and not FORCE_REDOWNLOAD_DATASET:
    print("Dataset đã có sẵn tại:", RAW_DATASET_DIR, "- bỏ qua bước tải.")
else:
    clone_dir = "/content/DeepCrack_repo"
    if os.path.exists(clone_dir):
        !rm -rf {clone_dir}
    !git clone --depth 1 https://github.com/yhlleo/DeepCrack.git {clone_dir}

    import shutil, zipfile
    src_zip = Path(clone_dir) / "dataset" / "DeepCrack.zip"
    assert src_zip.exists(), f"Không thấy {src_zip}, cấu trúc repo DeepCrack có thể đã đổi."

    with zipfile.ZipFile(src_zip) as z:
        z.extractall(RAW_DATASET_DIR)

    assert extracted_marker.exists(), "Giải nén xong nhưng không thấy train_img/ - kiểm tra lại cấu trúc zip."
    print("Đã giải nén dataset vào:", RAW_DATASET_DIR)

for split in ["train_img", "train_lab", "test_img", "test_lab"]:
    p = Path(RAW_DATASET_DIR) / split
    n = len(list(p.glob("*"))) if p.exists() else 0
    print(f"{split}: {n} file")

In [ ]:
# =============================
# 6. Convert mask PNG -> nhãn YOLO segmentation (polygon), chia train/val, giữ nguyên test set riêng
# =============================
import cv2, os, random, shutil, yaml
from pathlib import Path

data_yaml_exists = Path(DATA_YAML_PATH).exists()
if data_yaml_exists and not FORCE_RECONVERT_LABELS:
    print("Đã có sẵn YOLO dataset đã convert tại:", YOLO_DATASET_DIR, "- bỏ qua bước convert.")
else:
    if Path(YOLO_DATASET_DIR).exists():
        shutil.rmtree(YOLO_DATASET_DIR)

    def mask_to_yolo_polygons(mask_path, min_area=6, epsilon_ratio=0.002):
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        h, w = mask.shape[:2]
        _, binary = cv2.threshold(mask, 30, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        polygons = []
        for c in contours:
            if cv2.contourArea(c) < min_area:
                continue
            peri = cv2.arcLength(c, True)
            approx = cv2.approxPolyDP(c, epsilon_ratio * peri, True)
            if len(approx) < 3:
                continue
            pts = approx.reshape(-1, 2).astype(float)
            pts[:, 0] /= w
            pts[:, 1] /= h
            polygons.append(pts.flatten().tolist())
        return polygons

    def build_split(img_dir, lab_dir, basenames, out_img_dir, out_lab_dir):
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_lab_dir.mkdir(parents=True, exist_ok=True)
        n_with_crack = 0
        for name in basenames:
            src_img = img_dir / f"{name}.jpg"
            src_lab = lab_dir / f"{name}.png"
            if not src_img.exists() or not src_lab.exists():
                continue
            shutil.copy(src_img, out_img_dir / f"{name}.jpg")
            polygons = mask_to_yolo_polygons(src_lab)
            if polygons:
                n_with_crack += 1
            with open(out_lab_dir / f"{name}.txt", "w") as f:
                for poly in polygons:
                    coords = " ".join(f"{v:.6f}" for v in poly)
                    f.write(f"0 {coords}\n")
        return n_with_crack

    raw = Path(RAW_DATASET_DIR)
    train_img_dir = raw / "train_img"
    train_lab_dir = raw / "train_lab"
    test_img_dir = raw / "test_img"
    test_lab_dir = raw / "test_lab"

    all_train_names = sorted(p.stem for p in train_img_dir.glob("*.jpg"))
    random.Random(42).shuffle(all_train_names)
    n_val = max(1, int(len(all_train_names) * VAL_SPLIT_RATIO))
    val_names = all_train_names[:n_val]
    train_names = all_train_names[n_val:]
    test_names = sorted(p.stem for p in test_img_dir.glob("*.jpg"))

    yolo_root = Path(YOLO_DATASET_DIR)
    stats = {}
    stats["train"] = build_split(train_img_dir, train_lab_dir, train_names,
                                  yolo_root / "images/train", yolo_root / "labels/train")
    stats["val"] = build_split(train_img_dir, train_lab_dir, val_names,
                                yolo_root / "images/val", yolo_root / "labels/val")
    stats["test"] = build_split(test_img_dir, test_lab_dir, test_names,
                                 yolo_root / "images/test", yolo_root / "labels/test")

    data_yaml = {
        "path": str(yolo_root),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "crack"},
    }
    with open(DATA_YAML_PATH, "w") as f:
        yaml.safe_dump(data_yaml, f, allow_unicode=True)

    print("Train:", len(train_names), "ảnh, trong đó có vết nứt:", stats["train"])
    print("Val:  ", len(val_names), "ảnh, trong đó có vết nứt:", stats["val"])
    print("Test: ", len(test_names), "ảnh, trong đó có vết nứt:", stats["test"])
    print("data.yaml ghi tại:", DATA_YAML_PATH)

In [ ]:
# =============================
# 7. Kiểm tra nhanh vài mẫu đã convert (vẽ lại polygon đè lên ảnh, so bằng mắt với mask gốc)
# =============================
import cv2, glob, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sample_imgs = random.Random(1).sample(
    sorted(glob.glob(f"{YOLO_DATASET_DIR}/images/train/*.jpg")), 6
)

plt.figure(figsize=(16, 8))
for i, img_path in enumerate(sample_imgs):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lab_path = Path(YOLO_DATASET_DIR) / "labels/train" / (Path(img_path).stem + ".txt")
    overlay = img.copy()
    if lab_path.exists():
        with open(lab_path) as f:
            for line in f:
                vals = list(map(float, line.split()[1:]))
                pts = np.array(
                    [(int(vals[j] * w), int(vals[j + 1] * h)) for j in range(0, len(vals), 2)],
                    dtype=np.int32,
                )
                cv2.polylines(overlay, [pts], True, (255, 0, 0), 2)
    plt.subplot(2, 3, i + 1)
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(Path(img_path).name)
plt.tight_layout()
plt.show()

In [ ]:
# =============================
# 8. Train YOLOv8-seg -> lưu vào Drive (TRAIN_PROJECT_DIR), resume nếu bị ngắt giữa chừng
# =============================
from ultralytics import YOLO
from pathlib import Path

best_ckpt = Path(TRAIN_PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
last_ckpt = Path(TRAIN_PROJECT_DIR) / RUN_NAME / "weights" / "last.pt"

if best_ckpt.exists() and not FORCE_RETRAIN:
    print("Đã train xong trước đó, bỏ qua bước train:", best_ckpt)
    print("Bật FORCE_RETRAIN = True nếu muốn train lại từ đầu.")
    model = YOLO(str(best_ckpt))
elif last_ckpt.exists() and not FORCE_RETRAIN:
    print("Phát hiện checkpoint dang dở, RESUME training từ:", last_ckpt)
    model = YOLO(str(last_ckpt))
    model.train(resume=True)
else:
    print("Train từ pretrained:", BASE_MODEL)
    model = YOLO(BASE_MODEL)
    model.train(
        data=DATA_YAML_PATH,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        project=TRAIN_PROJECT_DIR,
        name=RUN_NAME,
        exist_ok=True,
        patience=30,
    )

print("Best weight:", best_ckpt)

In [ ]:
# =============================
# 9. Đánh giá trên test set (hoàn toàn tách biệt, không dùng lúc train)
# =============================
from ultralytics import YOLO
from pathlib import Path

best_ckpt = Path(TRAIN_PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best_ckpt))

metrics = model.val(data=DATA_YAML_PATH, split="test")
print("mAP50 (mask):", metrics.seg.map50)
print("mAP50-95 (mask):", metrics.seg.map)

In [ ]:
# =============================
# 10. Export sang ONNX -> lưu vào Drive, dùng để chạy trực tiếp trên trình duyệt (docs/crack/)
# =============================
from ultralytics import YOLO
from pathlib import Path
import shutil

best_ckpt = Path(TRAIN_PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
onnx_out = Path(TRAIN_PROJECT_DIR) / RUN_NAME / "weights" / "best.onnx"

if onnx_out.exists() and not FORCE_RETRAIN:
    print("Đã có sẵn file ONNX, bỏ qua export:", onnx_out)
else:
    model = YOLO(str(best_ckpt))
    # opset 12 để tương thích rộng với onnxruntime-web; simplify=True giúp model gọn hơn, chạy nhanh hơn trên web.
    # imgsz cố định (không dynamic) vì web build đơn giản hơn nhiều khi input shape cố định.
    exported_path = model.export(format="onnx", imgsz=IMG_SIZE, opset=12, simplify=True, dynamic=False)
    exported_path = Path(exported_path)
    if exported_path != onnx_out:
        shutil.copy(exported_path, onnx_out)

print("ONNX model tại:", onnx_out)
print("Size MB:", onnx_out.stat().st_size / 1024 / 1024)
print("Tải file này về, đặt vào docs/crack/model/best.onnx trong repo để web dùng.")

from google.colab import files
files.download(str(onnx_out))

In [ ]:
# =============================
# 11. Demo trên vài ảnh test - xem trực quan mask dự đoán
# =============================
from ultralytics import YOLO
from pathlib import Path
import glob, random
import matplotlib.pyplot as plt

best_ckpt = Path(TRAIN_PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best_ckpt))

sample_imgs = random.Random(2).sample(
    sorted(glob.glob(f"{YOLO_DATASET_DIR}/images/test/*.jpg")), 6
)
results = model.predict(sample_imgs, conf=0.25, save=False)

plt.figure(figsize=(16, 8))
for i, r in enumerate(results):
    plt.subplot(2, 3, i + 1)
    plt.imshow(r.plot()[:, :, ::-1])
    plt.axis("off")
    plt.title(Path(r.path).name)
plt.tight_layout()
plt.show()

In [ ]:
# =============================
# 12. Chạy trên ảnh thật của bạn (upload trực tiếp) - đây là bước "ứng dụng" thực tế
# =============================
from google.colab import files
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import cv2

best_ckpt = Path(TRAIN_PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best_ckpt))

print("Chọn 1 hoặc nhiều ảnh chụp vật thể/công trình có vết nứt để test:")
uploaded = files.upload()

for fname in uploaded.keys():
    result = model.predict(fname, conf=0.25, save=False)[0]
    out_path = Path(DEMO_OUTPUT_DIR) / f"pred_{fname}"
    cv2.imwrite(str(out_path), result.plot())
    print("Đã lưu kết quả:", out_path)

    plt.figure(figsize=(8, 8))
    plt.imshow(result.plot()[:, :, ::-1])
    plt.axis("off")
    plt.title(fname)
    plt.show()

## Ghi chú - áp dụng weight này vào đâu tiếp theo

- Weight tốt nhất nằm tại `TRAIN_PROJECT_DIR/{RUN_NAME}/weights/best.pt` trên Google Drive - tải về hoặc
  dùng thẳng từ Drive cho các bước sau.
- **Dùng ngay cho demo (đã sẵn sàng):** cell 11/12 ở trên - chạy trên ảnh/video thật của vật thể sẽ quay
  cho pipeline 3D, cho ra ảnh có khoanh vùng vết nứt.
- **Demo trên web (cell 10 export ONNX):** tải file `best.onnx` về, đặt vào `docs/crack/model/best.onnx`
  trong repo - trang `docs/crack/` chạy model thẳng trên trình duyệt bằng `onnxruntime-web`, không cần
  server/GPU, người xem tự upload ảnh và xem kết quả ngay tại chỗ.
- **Tích hợp với pipeline 3D (bước tiếp theo, chưa làm trong notebook này):** chạy model này trên đúng tập
  frame đã dùng để train 3D Gaussian Splatting (`INPUT_DIR` trong `heritage_twin_pipeline_colab.ipynb`),
  sau đó dùng camera pose COLMAP đã tính sẵn (`scene/colmap_loader.py` trong repo `gaussian-splatting`) để
  chiếu tâm các Gaussian vào từng frame và đánh dấu điểm nào rơi vào vùng mask vết nứt - từ đó tô màu vết
  nứt ngay trên mô hình 3D thay vì chỉ trên ảnh 2D rời rạc.